# Extraction of Spectral Features and Transition Probabilities

### Import all the necessary packages and modules.

In [ ]:
from itertools import groupby

import librosa
import librosa.display
import matplotlib.pyplot as plt
import mir_eval
import numpy as np
import scipy
import soundfile
from glob import glob
import pandas as pd
from time import sleep

### Load the hums as wav audio files.

In [ ]:
# Specify directory and local paths to all improv songs of a participant.

directory = glob("C:/Users/monik/Documents/Paris/Internship/Analyses/Pilot-Data/999/IMPROV/*.wav")
                                                                             # XXX = PARTICIPANT CODE
                                                                             # YYY = CONDITION (IMPROV/Song-Forms/HB/WU)
datapaths = []
filenames = []

for el in directory:
    split_el = el.split("\\")                   
    datapaths.append(split_el[0]) # append the first element which is the local path
    filenames.append(split_el[1]) # append the second element which is the name of the audio file without the local path

# Sort the songs in order from 1st to 30th
filenames.sort()

# Read in the audio files

datapath = datapaths[0]
datapath = datapath + ('/')

In [ ]:
# Quick check for order of songs.
print(datapath)
print(len(filenames))
print(filenames)

### Extract relevant spectral features.

#### Load audio files to librosa.

In [ ]:
# Load the audio files to librosa for analysis.
y_all = []
sr_all = [] # Fixed to 22050 for all songs.

for filename in filenames:
    filepath = f'{datapath}{filename}'
    y, sr = librosa.load(filepath)
    y_all.append(y)
    sr_all.append(sr)
    
# Compute the spectrogram.
    D = librosa.stft(y)

# Convert an amplitude spectrogram to dB-scaled SPECTROGRAM.
    S_db = librosa.amplitude_to_db(np.abs(D), ref=np.max) 

In [ ]:
# Check the sampling rate and the length of y_all. There should be 30 y-s for IMPROV and 16 for SONG-FORM 
print(len(y_all))
print(np.unique(sr_all))

#### Extract timeframe, fundamental frequency, and amplitude per song.

In [ ]:
# Initialize empty list for lopping through all songs.
sr = sr_all[0] # Constant for all songs.
tf_all = [] # Time frame for all songs per participant.
f0_all = [] # Fundamental frequency for all songs per participant.

# Set a progress bar that will tell you at which song the program is and how much is left.
length_bar = len(y_all)
bar = '-'*length_bar
replace_no = len(y_all)/length_bar

for i in range(0, len(y_all)): # Loop through all songs of this participant to get f0 of each song.
    
    # Print a progress bar.
    num_squares = int(i/replace_no)
    bar = '#'*num_squares+'-'*(length_bar-num_squares)
    percentage = round((i*100)/(len(y_all)-1), 2) # round to 2 int
    sleep(.1)
    print(bar, percentage, '%', end='\r')
    
    # Get f0.
    f0, vid, vpd = librosa.pyin(
        y_all[i],
        sr=sr,
        fmin=librosa.note_to_hz("C1"), # The min and max are restrictive - lower & higher values are feasible.
                                       # The recommended min value = C1, recommended max value = C7.
        fmax=librosa.note_to_hz("C7"),
        resolution=0.1, # 0.01 corresponds to cent resolution but it takes longer, change if necessary for other analyses
    )
    f0_all.append(f0)
    

    # Create a numpy array for time frames.
    tf = np.linspace(start=0, stop =(len(y_all[i]) / sr), num=len(f0))
    tf_all.append(tf)

#### Extract chroma.

In [ ]:
# Initialize lists of np.arrays for all songs for a participant.
sr = sr_all[0] # Constant for all songs.
chroma_all = []
chroma_filter_all = []
kernel_all = []
chroma_smooth_all = []
chroma_thresh_all = []
chroma_index_all = []

# Set a progress bar.
length_bar = len(y_all)
bar = '-'*length_bar
replace_no = len(y_all)/length_bar

# Loop all songs of this participant to get chroma of each song.
for i in range(0, len(y_all)):
    
    # Progress bar
    num_squares = int(i/replace_no)
    bar = '#'*num_squares+'-'*(length_bar-num_squares)
    percentage = round((i*100)/(len(y_all)-1), 2)
    sleep(.1)
    print(bar, percentage, '%', end='\r')
    
    # Optimize filtering
    chroma = librosa.feature.chroma_cqt(y=y_all[i], sr=sr, norm=None)  # In CQT chromagram
                                                                       # (Constant-Q
                                                                       # Tranform) one 
                                                                       # vertical step
                                                                       # represents one
                                                                       # semitone
    chroma_all.append(chroma)
    
    # Sparse de-noising of the chromagram: remove noise in the signal such as vibrato.
    chroma_filter = np.minimum(
        chroma, librosa.decompose.nn_filter(chroma, aggregate=np.median, metric="cosine")
    )
    chroma_filter_all.append(chroma_filter)

    # Construct a small convolution matrix 
        # First, contruct an array by repeating the first argument the number of times given by the second argument.
    kernel = np.tile(np.array([-0.5, 1, -0.5])[:, None], (1, 9)) # Assign max value to the middle and smooth out the neighbours.
        # Second, divide the matrix with the absolute values of itself that are summed. 
    kernel /= np.abs(kernel).sum() # kernel / np.abs(kernel).sum()
    kernel_all.append(kernel)

    # Multidimensional convolution between the kernel and the chroma
    chroma_smooth = scipy.ndimage.filters.convolve(chroma_filter, kernel)
                # Here, chroma_smooth is a numpy array of 12 sub-numpy arrays that each have 1290 values.
                # In Western music, chroma reflects the twelve different pitch classes: C, C♯, D, D♯, E , F, F♯, G, G♯, A, A♯, B
    chroma_smooth_all.append(chroma_smooth)

    # Threshold
    chroma_thresh = chroma_smooth
    chroma_thresh[chroma_thresh < 0.1] = -np.inf # Values smaller than 0.1 become a negative numpy infinity. Here we
                                                 # exclude the predictions that the chroma makes for silences in the audio.
    chroma_thresh_all.append(chroma_thresh)
    
    # Index of chroma: 0 - 11
    chroma_index = np.argmax(chroma_thresh, axis=0).astype(float) # Take the maximum probability of the chroma in each window.
    nanid = np.all(np.isinf(chroma_thresh), axis=0) # Test whether all array elements along a given axis evaluate to True.
    chroma_index[nanid] = np.nan
    inList = list()
    
    for k, g in groupby(chroma_index): # Groupby all the samples per window. K is the index of chroma in the list of Western
                                       # pitch scale. K has a range from 0 to 11: C, C♯, D, D♯, E , F, F♯, G, G♯, A, A♯, B.
        seg = [*g]
        if len(seg) > 3: # 3 segments correspond to about 60 ms - might cut of participant that change pitch fast purposely!
            inList += seg
        else:
            inList += [np.nan] * len(seg)

    chroma_index = np.array(inList) # The second index is a corrected version of the first index variable.
    chroma_index_all.append(chroma_index)

In [ ]:
# Tests for chroma index values (due to thresholding & segment selection).
print(len(chroma_index_all[0]))
print(len(chroma_index_all))
print(type(chroma_index_all[6]))
print(np.unique(chroma_index_all[9]))

# print(len(chroma_tresh_all[0][0])) # Chroma treshold has N lists (N = number of songs) with 12 sublists.
                                     # The sublists have information about probabilities of each of the 12 chroma occurance.
                                     # Each sublist has len(y_all) values like tf variable.
# print(type(chroma_tresh_all[1]))

#### Extract hz, midi_notes, and note names (math octaves maintaining duration).

In [ ]:
# Initialize empty lists for looping.
hz_all = []
midi_notes_all =[]
note_names_all = []

# Set a progress bar.
length_bar = len(y_all)
bar = '-'*length_bar
replace_no = len(y_all)/length_bar

# Get Hz and Midi Notes for all songs.
for i in range(0,len(y_all)):
    
    # Get note information.
    index_labels = ["C", "C#", "D", "D#", "E", "F", "F#", "G", "G#", "A", "A#", "B"]
    
    hz = list()
    midi_notes = list()
    note_names = list()

    for k, g in groupby(zip(chroma_index_all[i], f0_all[i]), key=lambda x: x[0]):
        freqs = [t[1] for t in g]
        if np.isnan(k):
            hz.append(k)
            midi_notes.append(k)
            note_names.append(k)
#         elif len(freqs) == 0:
#             hz.append(-1)
#             midi_notes.append(-1)
#             note_names.append(-1)
        else:

            # Try to identify which note and which octave.
            currlab = index_labels[int(k)] # Find the pitch letter by index of [C, C♯, D, D♯, E , F, F♯, G, G♯, A, A♯, B].
            posF = librosa.note_to_hz([f"{currlab}{i}" for i in range(1, 7)])
            avF = np.nanmean(freqs)
            octI = np.argmin(np.abs(posF - avF)) + 1 # Integer value that indicates the octave.
            note_name = f"{currlab}{octI}"
            
            # Convert to Hz and Midi
            freq = librosa.note_to_hz(note_name)
            midi = librosa.note_to_midi(note_name)
            validSamp = ~np.isnan(freqs)
            hz += list(freq * validSamp)
            midi_notes += [midi if vS else np.nan for vS in validSamp]
            note_names += [note_name if vS else np.nan for vS in validSamp]

    # Put into numpy array
    hz = np.array(hz)
    hz[np.isnan(hz)] = -1
    
    # Put into the all-songs list.
    hz_all.append(hz)
    midi_notes_all.append(midi_notes)
    note_names_all.append(note_names)
    
    # Show progress bar.
    num_squares = int(i/replace_no)
    bar = '#'*num_squares+'-'*(length_bar-num_squares)
    percentage = round((i*100)/(len(y_all)-1), 2)
    sleep(.1)
    print(bar, percentage, '%', end='\r')

In [ ]:
# Tests

#print(hz_all[0])
#print(midi_notes_all[0])
#print(note_names_all[0])

print(len(hz_all[0]))
print(len(midi_notes_all[0]))
print(len(note_names_all[0]))

# print(type(hz_all[1])) # NUMPY ARRAY
# print(type(midi_notes_all[1])) # LIST
# print(type(note_names_all[0])) # LIST

In [ ]:
# # Are they correctly matched? YES.
print(len(tf_all[3]))
print(len(hz_all[3]))
print(len(f0_all[3]))
print(len(chroma_index_all[3]))
print(len(midi_notes_all[3]))
print(len(note_names_all[3]))
print(' ')

print(len(tf_all[6]))
print(len(hz_all[6]))
print(len(f0_all[6]))
print(len(chroma_index_all[6]))
print(len(note_names_all[6]))

print(' ')

print(len(tf_all[1]))
print(len(hz_all[1]))
print(len(f0_all[1]))
print(len(chroma_index_all[1]))
print(len(note_names_all[1]))


#### Extract phase information and transition probabilities.

In [ ]:
# Initialize empty lists for looping.
fphase_all = []
cphase_all = []

sequence_all = []

trans_all = []
med_durations_all = []

XX_all = [] # Contains pairs of current and previous pitch phases for all songs.
LW_all = []
COLOR_all = []

values_all = []
size_all = []
vPh_all = []

# Set a progress bar.
length_bar = len(y_all)
bar = '-'*length_bar
replace_no = len(y_all)/length_bar

# Run the loop
for u in range(len(y_all)):
    
    # Show progress bar.
    num_squares = int(u/replace_no)
    bar = '#'*num_squares+'-'*(length_bar-num_squares)
    percentage = round((u*100)/(len(y_all)-1), 2)
    sleep(.1)
    print(bar, percentage, '%', end='\r')
    
    # Phase information
    fphase = 2 * np.pi * (f0_all[u] - 125) / 125 # Numpy array of the fundamental frequency phases (len(fphase) == len(y_all)).
    fphase_all.append(fphase)
    
    cphase = 2 * np.pi * chroma_index_all[u] / 12 # Numpy array of the chroma phases (len(cphase) == len(y_all)).
    cphase_all.append(cphase)
    
    # Getting transition probabilites from a sequence of chroma values. Sequence is a list of sublists. 
    # The sublists have two integer values.
    sequence = np.array([[k, len([*g])] for k, g in groupby(chroma_index_all[u][~np.isnan(chroma_index_all[u])])
                        ]).astype(int)
    if len(sequence) == 0: ########################### ----------------------------------------------------- ????????
        sequence = np.zeros([2, 2]).astype(int)
        sequence_all.append(sequence)
    else:
        sequence_all.append(sequence)
    # Initialize an array for transition probabilities between notes. Current pitch (12) and Previous pitch (12).
    trans = np.zeros([12, 12])

    for i in range(1, len(sequence)):
        curr, prev = sequence[i, 0], sequence[i - 1, 0] # Current = curr, Previous = prev. 
                                                        # These values specify the location in the trans matrix.
        trans[prev, curr] += 1
    
    trans_all.append(trans)
    
    # Initialize the array for the median durations of the trans pair.
    med_durations = np.zeros([12, 12])
    
    for prev in range(12):
        for curr in range(12):
            # get median time length for each
            select = np.insert(
                (sequence[:-1, 0] == prev) & (sequence[1:, 0] == curr), 0, False
            ) # if False, insert nan. If True, insert value.
            med_durations[prev, curr] = np.median(sequence[select, 1])
    
    med_durations_all.append(med_durations)
    
    # Initialize list variables to store phase informaiton for each value of the trans pair.
    vmax = 30.0 #
    XX = list() #
    LW = list() # WHAT IS THIS INFORMATION ? ----------------------------------------------------- ????????
    COLOR = list() # WHAT IS THIS INFORMATION ? -------------------------------------------------- ????????

    for prev in range(12):
        pPh = 2 * np.pi * prev / 12 # Phase of the previous tone
        for curr in range(12):
            cPh = 2 * np.pi * curr / 12 # Phase of the current tone
            thisTrans = trans[prev, curr]
            thisDur = med_durations[prev, curr]
            if thisTrans > 0:
                XX.append([pPh, cPh])
                LW.append([6 * thisTrans / trans.max()])
                COLOR.append(np.minimum(thisDur / vmax, 1.0))
    
    XX_all.append(XX)
    LW_all.append(LW)
    COLOR_all.append(COLOR)
    
    # Get values corresponding to values from 0 to 11 that have been sung in the specific song.
    # The size corresponds to the amount of times that chroma value was sung.
    
    values, size = np.unique(sequence[:, 0], return_counts=True)
    values_all.append(values)
    size_all.append(size)
    
    # Get the phase of the values.
    vPh = 2 * np.pi * values / 12
    vPh_all.append(vPh)

#### Keep track of octaves for transition probabilites.

In [ ]:
# Initialize empty list variables.
note_names_all_2 = []
midi_index_all = []
semitones_all = []
semitrans_all = []

# Start the loop
for u in range(len(y_all)):
    
    # Show progress bar.
    num_squares = int(u/replace_no)
    bar = '#'*num_squares+'-'*(length_bar-num_squares)
    percentage = round((u*100)/(len(y_all)-1), 2)
    sleep(.1)
    print(bar, percentage, '%', end='\r')
    
    # For each note find the mean f0 value to best estimate absolute interval. The same procedure as for hz and midi extraction.
    index_labels = ["C", "C#", "D", "D#", "E", "F", "F#", "G", "G#", "A", "A#", "B"]
    note_names_2 = list()

    for k, g in groupby(
        zip(chroma_index_all[u][~np.isnan(chroma_index_all[u])], f0_all[u][~np.isnan(chroma_index_all[u])]),
        key=lambda x: x[0]
    ):
        freq = [t[1] for t in g]
        nSamps = sum(~np.isnan(freq))

        if nSamps > 0:
            currlab = index_labels[int(k)]
            posF = librosa.note_to_hz([f"{currlab}{i}" for i in range(1, 7)])
            avF = np.nanmean(freq)
            octI = np.argmin(np.abs(posF - avF)) + 1
            note_names_2.append(f"{currlab}{octI}")
            
    note_names_2 = np.array(note_names_2)
    midi_index = librosa.note_to_midi(note_names_2)
    semitones = np.diff(midi_index) # Differences between tones.
    
    note_names_all_2.append(note_names_2)
    midi_index_all.append(midi_index)
    semitones_all.append(semitones)
    
    # Extract the transition probabilities between pitch intervals (distances between neighbouring pitches)
    if len(semitones) == 0:
        semitones = np.nan
    else:
        xy = range(np.min([-12, semitones.min()]), np.max([13, semitones.max() + 1]))
        semitrans = np.zeros([len(xy)] * 2)

        for i in range(1, len(semitones)):
            curr, prev = semitones[i], semitones[i - 1]
            semitrans[xy.index(prev), xy.index(curr)] += 1.0

        semitrans[semitrans == 0] = np.nan
        # semitrans /= np.nansum(semitrans, axis=1, keepdims=True)
        semitrans /= np.nansum(semitrans) # The final difference between the intervals.
        semitrans_all.append(semitrans)

In [ ]:
#Compare note names for both steps: Hz-Midi & Transition-Probabilities ########################## COMPARISON !!!!!!!
# print(note_names_all_2[1]-note_names_all[1])
# print(note_names_all_2[1])
# print(np.unique(np.array(note_names_all[1])))

In [ ]:
# # Checks
# s = []
# n = []
# m = []
# for i in range(len(y_all)):
#     s.append(len(sequence_all[i])) # variable
#     n.append(len(note_names_all_2[i]))
#     m.append(len(midi_index_all[i]))

# print(m == n)

# print(len(semitones_all[22]))
# print(len(semitrans_all[22]))

#### Save extracted variables into data frames.

##### Spectral features

In [ ]:
# Initialize empty dataframe for spectral features.
df_spectral = pd.DataFrame()

# Add columns with the extracted spectral features
df_spectral['TIME_FRAME'] = pd.DataFrame(data=np.concatenate(tf_all))
df_spectral['PITCH'] = pd.DataFrame(data=np.concatenate(hz_all))
df_spectral['FUNDAMENTAL_FREQUENCY'] = pd.DataFrame(data=np.concatenate(f0_all))
df_spectral['CHROMA_ID'] = pd.DataFrame(data=np.concatenate(chroma_index_all))
df_spectral['CHROMA_OCTAVE'] = pd.DataFrame(data=np.concatenate(note_names_all))
df_spectral['MIDI_NOTE'] = pd.DataFrame(data=np.concatenate(midi_notes_all))
df_spectral['F0_phase'] = pd.DataFrame(data=np.concatenate(fphase_all))
df_spectral['CHROMA_phase'] = pd.DataFrame(data=np.concatenate(cphase_all))

# Extract participant code.
participant_ID = []

for filename in filenames:
    split_first = filename.split('_recording_mic_')
    participant_ID.append(split_first[0])

participant_ID_spectral = participant_ID*len(np.concatenate(tf_all))

# Add a column with the ID of the participant.
df_spectral['PARTICIPANT_ID'] = pd.DataFrame(data=participant_ID_spectral)

# Add song number information. # Code for Improv conditions
song_ID = []
for count, sample in enumerate(tf_all, 1): # Start the counter with 1 instead of 0.
    len_sample = len(sample)
    song_no = [count]*len_sample
    song_ID.append(song_no)

song_ID = np.concatenate(song_ID)

df_spectral['SONG_ID'] = pd.DataFrame(data=song_ID)

# Add song number information. # Code for Song-form conditions
song_ID = []
for count, sample in enumerate(tf_all, 1): # Start the counter with 1 instead of 0.
    len_sample = len(sample)
    song_no = [count]*len_sample
    if count <= 4:
        for el in list(song_no):
            song_no = 'Dance'+'_'+str(el)
            song_ID.append(song_no)
    elif count <= 8:
        for el in list(song_no):
            song_no = 'Healing'+'_'+str(el-4)
            song_ID.append(song_no)
    elif count <= 12:
        for el in list(song_no):
            song_no = 'Love'+'_'+str(el-8)
            song_ID.append(song_no)
    else:
        for el in list(song_no):
            song_no = 'Lullaby'+'_'+str(el-12)
            song_ID.append(song_no)

df_spectral['SONG_ID'] = pd.DataFrame(data=song_ID)

# Tests
df_spectral.info()
df_spectral.head()

In [ ]:
# Check the SONG_ID columns
print(df_spectral.head())
print(df_spectral['SONG_ID'].unique())

##### Transition probabilities - Notes

In [ ]:
# Create a dataframe for transitional probabilities between notes
df_transitions = pd.DataFrame()
for i in range(len(trans_all)):
    df_i = pd.DataFrame(trans_all[i], columns = ["C", "C#", "D", "D#", "E", "F", "F#", "G", "G#", "A", "A#", "B"])
    df_transitions = pd.concat([df_transitions, df_i], axis=0, ignore_index = True)

# Add participant ID.
participant_ID_transitions = participant_ID*(len(tf_all)*12) # Twelve rows for each song.
df_transitions['PARTICIPANT_ID'] = pd.DataFrame(data=participant_ID_transitions)

# Add song number information. # Code for Improv conditions
song_ID = []
for count, sample in enumerate(trans_all, 1): # Start the counter with 1 instead of 0.
    len_sample = len(sample)
    song_no = [count]*len_sample
    song_ID.append(song_no)

song_ID = np.concatenate(song_ID)

df_transitions['SONG_ID'] = pd.DataFrame(data=song_ID)

# Add song number information. # Code for Song-form conditions
song_ID = []
for count, sample in enumerate(trans_all, 1): # Start the counter with 1 instead of 0.
    len_sample = len(sample)
    song_no = [count]*len_sample
    if count <= 4:
        for el in list(song_no):
            song_no = 'Dance'+'_'+str(el)
            song_ID.append(song_no)
    elif count <= 8:
        for el in list(song_no):
            song_no = 'Healing'+'_'+str(el-4)
            song_ID.append(song_no)
    elif count <= 12:
        for el in list(song_no):
            song_no = 'Love'+'_'+str(el-8)
            song_ID.append(song_no)
    else:
        for el in list(song_no):
            song_no = 'Lullaby'+'_'+str(el-12)
            song_ID.append(song_no)

df_transitions['SONG_ID'] = pd.DataFrame(data=song_ID)

# Tests
df_transitions.info()
df_transitions.head()

In [ ]:
# Check the SONG_ID columns
print(df_transitions.head())
print(df_transitions['SONG_ID'].unique())

In [ ]:
# # Create a dataframe for transitional probabilities between SEMITONES ?!!
# tp_semitones = pd.DataFrame()
# for i in range(len(semitones_all)):
#     df_i = pd.DataFrame(semitones_all[i])
#     tp_semitones = pd.concat([tp_semitones, df_i], axis=0, ignore_index = True)

# # Add participant ID.
# participant_ID_transitions = participant_ID*(len(tf_all)*12) # Twelve rows for each song.
# tp_semitones['PARTICIPANT_ID'] = pd.DataFrame(data=participant_ID_transitions)

# # # Add song number information. # Code for Improv conditions
# # song_ID = []
# # for count, sample in enumerate(trans_all, 1): # Start the counter with 1 instead of 0.
# #     len_sample = len(sample)
# #     song_no = [count]*len_sample
# #     song_ID.append(song_no)

# # song_ID = np.concatenate(song_ID)

# # df_transitions['SONG_ID'] = pd.DataFrame(data=song_ID)

# # Add song number information. # Code for Song-form conditions
# song_ID = []
# for count, sample in enumerate(semitones_all, 1): # Start the counter with 1 instead of 0.
#     len_sample = len(sample)
#     song_no = [count]*len_sample
#     if count <= 2:
#         for el in list(song_no):
#             song_no = 'Dance'+'_'+str(el)
#             song_ID.append(song_no)
#     elif count <= 4:
#         for el in list(song_no):
#             song_no = 'Healing'+'_'+str(el-2)
#             song_ID.append(song_no)
#     elif count <= 6:
#         for el in list(song_no):
#             song_no = 'Love'+'_'+str(el-4)
#             song_ID.append(song_no)
#     else:
#         for el in list(song_no):
#             song_no = 'Lullaby'+'_'+str(el-6)
#             song_ID.append(song_no)

# tp_semitones['SONG_ID'] = pd.DataFrame(data=song_ID)

# # Tests
# tp_semitones.info()
# tp_semitones.head()

##### Transition probabilities - Intervals between Notes

In [ ]:
# print(len(semitrans_all)) #### 6????!!!
# # print(type(semitrans_all))

In [ ]:
# # Semitrans
# # Create a dataframe for transitional probabilities between SEMITRANS ?!!
# tp_semitrans = pd.DataFrame()
# for i in range(len(semitrans_all)):
#     df_i = pd.DataFrame(semitrans_all[i])
#     tp_semitrans = pd.concat([tp_semitrans, df_i], axis=0, ignore_index = True)

# # Add participant ID.
# participant_ID_transitions = participant_ID*(len(tf_all)*12) # Twelve rows for each song.
# tp_semitrans['PARTICIPANT_ID'] = pd.DataFrame(data=participant_ID_transitions)

# # # Add song number information. # Code for Improv conditions
# # song_ID = []
# # for count, sample in enumerate(trans_all, 1): # Start the counter with 1 instead of 0.
# #     len_sample = len(sample)
# #     song_no = [count]*len_sample
# #     song_ID.append(song_no)

# # song_ID = np.concatenate(song_ID)

# # df_transitions['SONG_ID'] = pd.DataFrame(data=song_ID)

# Add song number information. # Code for Song-form conditions
song_ID = []
for count, sample in enumerate(semitrans_all, 1): # Start the counter with 1 instead of 0.
    len_sample = len(sample)
    song_no = [count]*len_sample
    if count <= 2:
        for el in list(song_no):
            song_no = 'Dance'+'_'+str(el)
            song_ID.append(song_no)
    elif count <= 4:
        for el in list(song_no):
            song_no = 'Healing'+'_'+str(el-2)
            song_ID.append(song_no)
    elif count <= 6:
        for el in list(song_no):
            song_no = 'Love'+'_'+str(el-4)
            song_ID.append(song_no)
    else:
        for el in list(song_no):
            song_no = 'Lullaby'+'_'+str(el-6)
            song_ID.append(song_no)

# tp_semitrans['SONG_ID'] = pd.DataFrame(data=song_ID)

# # Tests
# tp_semitrans.info()
# tp_semitrans.head()

#### Save all data frames as .csv files

In [ ]:
print(filename) # Copy the filename into the function below to get an appropriate .csv name !

In [ ]:
# Save all dataframes to csv.
df_spectral.to_csv(f"spectral_data_{filename.replace('_recording_mic_30_2023.wav', '_IMPROV.csv')}") # IMPROV or SONG-FORM

df_transitions.to_csv(f"trans_note_data_{filename.replace('_recording_mic_30_2023.wav', '_IMPROV.csv')}")

#tp_semitrans.to_csv(f"trans_interval_data_{filename.replace('_recording_mic_Lullaby_2023-03-21_15h56.16.785.wav', '_Song-Forms.csv')}")